In [1]:
import pandas as pd
from matplotlib import pyplot as plt
import os
import nrrd
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
from diffusers import DDIMScheduler, UNet2DModel
import torch
from tqdm import tqdm
from torch.optim import Adam
from datetime import datetime
import torch.nn as nn

device = torch.device("cuda")

/mnt/raid/home/ajarry/.conda/envs/difmod/lib/python3.10/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
/mnt/raid/home/ajarry/.conda/envs/difmod/lib/python3.10/site-packages/torchvision/transforms/v2/__init__.py:54: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Pl

In [6]:
# Read Parquet file into a DataFrame
parquet = "/mnt/raid/C1_ML_Analysis/CSV_files/extract_frames_Dataset_C_masked_resampled_256_spc075_wscores_meta_noflyto_1e-4.parquet"
df = pd.read_parquet(parquet, engine="pyarrow")
print(df)

                                                 img_path      study_id_x  \
0       extract_frames_blind_sweeps/Dataset_C_masked_r...  FAM-202-2445-1   
1       extract_frames_blind_sweeps/Dataset_C_masked_r...  FAM-202-2445-1   
2       extract_frames_blind_sweeps/Dataset_C_masked_r...  FAM-202-2445-1   
3       extract_frames_blind_sweeps/Dataset_C_masked_r...  FAM-202-2445-1   
4       extract_frames_blind_sweeps/Dataset_C_masked_r...  FAM-202-2445-1   
...                                                   ...             ...   
732786  extract_frames_blind_sweeps/Dataset_C_masked_r...      UNC-1038-2   
732787  extract_frames_blind_sweeps/Dataset_C_masked_r...      UNC-1038-2   
732788  extract_frames_blind_sweeps/Dataset_C_masked_r...      UNC-1038-2   
732789  extract_frames_blind_sweeps/Dataset_C_masked_r...      UNC-1038-2   
732790  extract_frames_blind_sweeps/Dataset_C_masked_r...      UNC-1038-2   

           score        pred                                    id  \
0    

We need 'manufacturer' == 'Butterfly Network Inc' or 'Butterfly Network Inc.'

In [3]:
image_data = df.iloc[8985,0]  # Adjust column name

data, _ = nrrd.read('/mnt/raid/C1_ML_Analysis/simulated_data_export/placenta_simu/FAM-025-0351-3_label11/M.nrrd')
print(data.shape)

image = Image.fromarray(data).convert('RGB')
image = image.rotate(270)*255

image

(256, 256, 200)


TypeError: Cannot handle this data type: (1, 1, 200), <f4

In [5]:
name = df.iloc[2,0]
print(df.shape[0])

732791


In [4]:
root_dir = '/mnt/raid/C1_ML_Analysis'

In [5]:
class DatasetFromDataFrame(Dataset):
    def __init__(self, root_dir, dataframe, transform=None):
        self.root_dir = root_dir
        self.df = dataframe
        self.transform = transform
        self.image_paths = []
        self.labels = []

        for i in range(df.shape[0]):
            img_name = df.iloc[i,0]
            self.image_paths.append(os.path.join(self.root_dir,img_name))
            self.labels.append(0)
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        data, header = nrrd.read(img_path)
        data = np.squeeze(data)
        normalized = (data - np.min(data)) / (np.max(data) - np.min(data) + 1e-5)
        scaled_data = (normalized * 255)
        scaled_data = scaled_data[0].astype(np.uint8)
        image = Image.fromarray(scaled_data).convert('RGB')
        image = image.rotate(270)
    

        if self.transform:
            image = self.transform(image)
        
        label = int(self.labels[idx])

        return image, label

In [6]:
transform = transforms.Compose([
    transforms.ToTensor(),           # Convert images to PyTorch tensors
])

In [7]:
train_dataset = DatasetFromDataFrame(root_dir=root_dir,dataframe=df,transform=transform)

In [ ]:
test_dataloader = DataLoader(train_dataset, batch_size=1, shuffle=True)
x, y = next(iter(test_dataloader))
print(x.size())
fig, axs = plt.subplots(1,1)
plt.imshow(torchvision.utils.make_grid(x)[0], cmap='gray')
plt.axis('off')

In [ ]:
import torch
from diffusers import DDIMScheduler
import matplotlib.pyplot as plt
import torchvision
from tqdm import tqdm

# Initialize the scheduler
scheduler = DDIMScheduler(num_train_timesteps=1000, beta_schedule="linear")

# Define the number of inference steps as an integer
num_inference_steps = 8
scheduler.set_timesteps(num_inference_steps)


# Collect noised versions
noisy_x = []
for i in range(x.shape[0]):
    for t in scheduler.timesteps:
        t_tensor = torch.tensor([t], device=x.device).long()
        noised = scheduler.add_noise(x[i:i+1], torch.randn_like(x[i:i+1]), t_tensor)
        noisy_x.append(noised)

# Stack the results
noisy_x = torch.cat(noisy_x, dim=0)

# Plot the result
fig, axs = plt.subplots(1, 1, figsize=(12, 9))
axs.imshow(torchvision.utils.make_grid(noisy_x).permute(1, 2, 0), cmap='gray')
plt.show()


In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"output_parquet/output_{timestamp}/"
os.makedirs(output_dir, exist_ok=True)

# Initialize model, scheduler, and optimizer
model = UNet2DModel()
model = model.to(device)
optimizer = Adam(model.parameters(), lr=1)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
scheduler = DDIMScheduler(num_train_timesteps=1000, beta_schedule="linear")


# Training loop
num_epochs = 1
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")
    model.train()
    for batch_idx, (x,y) in enumerate(tqdm(train_dataloader, desc="Training", leave=False)):
        x = x.to(device)
        # Sample noise and timesteps
        noise = torch.randn_like(x)
        noise = noise.to(device)
        timesteps = torch.randint(0, scheduler.config.num_train_timesteps, (x.size(0),))
        timesteps=timesteps.to(device)
        # Forward pass with noisy input
        noisy_images = scheduler.add_noise(x, noise, timesteps)
        noise_pred = model(noisy_images, timesteps).sample

        # Compute loss
        loss = torch.nn.functional.mse_loss(noise_pred, noise)
        
        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")
    checkpoint_filename = f"checkpoint_epoch_{epoch+1}.pth"
    checkpoint_path = os.path.join(output_dir, checkpoint_filename)

    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        }
torch.save(checkpoint, checkpoint_path)

print("Training completed.")
final_model_path = os.path.join(output_dir, "model.pth")
torch.save(model.state_dict(), final_model_path)
